In [ ]:
# QC, hashtag-condition assignment and velocity setup for the in-vitro timecourse.
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import palantir
import loompy as lp

In [ ]:
# Load data
adata = sc.read_h5ad("/home/yanjun2/scQC/option2_2_final.h5ad")

In [ ]:
f_loom_path_scenic = "/home/yanjun2/scenic/option2_2.loom"

In [ ]:
row_attrs = {
    "Gene": np.array(adata.var_names) ,
}
col_attrs = {
    "CellID": np.array(adata.obs_names) ,
    "nGene": np.array( np.sum(adata.X.transpose()>0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(adata.X.transpose() , axis=0)).flatten() ,
}
lp.create( f_loom_path_scenic, adata.X.transpose(), row_attrs, col_attrs)

In [ ]:
f_tfs = "/home/yanjun2/scenic/hs_hgnc_tfs.txt"

In [ ]:
!pip install dask-expr==0.5.3

In [ ]:
!pyscenic grn {f_loom_path_scenic} {f_tfs} -o adj.csv --num_workers 10

In [ ]:
adata = sc.read_h5ad("/home/yc/Desktop/scQC/b4filter.h5ad")
adata.layers["counts"] = adata.X.copy()
adata

In [ ]:
adata.obs['hash.ID'].unique().tolist()

In [ ]:
print(adata.obs['hash.ID'].value_counts())

In [ ]:
ldata1 = scv.read('/home/yc/Desktop/loom/R1_D1MDI_D1A.loom', cache=True)
ldata2 = scv.read('/home/yc/Desktop/loom/R1_D2MDI_D2A.loom', cache=True)
ldata3 = scv.read('/home/yc/Desktop/loom/R1_D3MDI_D3A.loom', cache=True)
ldata4 = scv.read('/home/yc/Desktop/loom/R1_D4MDI_D4A.loom', cache=True)
ldata5 = scv.read('/home/yc/Desktop/loom/R1_D5MDI_D5A.loom', cache=True)
ldata6 = scv.read('/home/yc/Desktop/loom/R1_DMo_D0M.loom', cache=True)

In [ ]:
def clean_cell_ids(ldata):
    ldata.obs.index = ldata.obs.index.str.replace(r'^.*:', '', regex=True)
    ldata.obs.index = ldata.obs.index.str.replace(r'x$', '', regex=True)
    return ldata
ldata1 = clean_cell_ids(ldata1)
ldata2 = clean_cell_ids(ldata2)
ldata3 = clean_cell_ids(ldata3)
ldata4 = clean_cell_ids(ldata4)
ldata5 = clean_cell_ids(ldata5)
ldata6 = clean_cell_ids(ldata6)

In [ ]:
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()
ldata3.var_names_make_unique()
ldata4.var_names_make_unique()
ldata5.var_names_make_unique()
ldata6.var_names_make_unique()

In [ ]:
ldata = ldata1.concatenate([ldata2, ldata3, ldata4, ldata5,
                            ldata6])

In [ ]:
adata.obs_names = adata.obs_names.str.split('-').str[0]

In [ ]:
adata = scv.utils.merge(adata, ldata)

In [ ]:
adata.obs['hash.ID'].unique().tolist()

In [ ]:
# Save result
adata.write("option2_1.h5ad")

In [ ]:
adata.X = adata.layers['counts'].copy() 

In [ ]:
MDIX_hash_ids = [
    '240316-Donor2-MoMac-Day1-MDIX', '240316-Donor2-MoMac-Day2-MDIX', 
    '240316-Donor2-MoMac-Day3-MDIX', '240316-Donor2-MoMac-Day4-MDIX', 
    '240316-Donor2-MoMac-Day5-MDIX']


MDI_hash_ids = [
    '240316-Donor2-MoMac-Day1-MDI', '240316-Donor2-MoMac-Day2-MDI', 
    '240316-Donor2-MoMac-Day3-MDI',
    '240316-Donor2-MoMac-Day4-MDI', '240316-Donor2-MoMac-Day5-MDI'
]
M_hash_ids = ['240316-Donor2-MoMac-Day-0-M']
adata.obs.loc[adata.obs['hash.ID'].isin(MDIX_hash_ids), 'condition'] = 'MDIX'

adata.obs.loc[adata.obs['hash.ID'].isin(MDI_hash_ids), 'condition'] = 'MDI'

adata.obs.loc[adata.obs['hash.ID'].isin(M_hash_ids), 'condition'] = 'Momac'

In [ ]:
adata.obs['condition'].unique().tolist()

In [ ]:
adata

In [ ]:
adata.X = adata.layers['counts'].toarray().copy()


In [ ]:
adata.X = adata.layers['counts'].copy()

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(adata, ["pct_counts_mt"], groupby = 'timepoint')

In [ ]:
print(adata.obs['hash.ID'].value_counts())

In [ ]:
adata

In [ ]:
# QC
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
#sc.pp.scrublet(adata)

In [ ]:
adata

In [ ]:
#adata = adata[adata.obs['pct_counts_mt'] < 30]

In [ ]:
adata

In [ ]:
# Normalise, HVG and PCA
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes = 5000 ,min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(adata, color = "hash.ID")

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 1,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_1'
)

In [ ]:
sc.pl.umap(adata, color=["leiden_res_1", "leiden_res_1_5","hash.ID", "condition", "timepoint"], legend_loc = 'right margin', ncols = 1
           #, save = 'umap.pdf'
          )

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res_1_5", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_res_1_5", standard_scale="var", n_genes=10, dendrogram = None, save = 'option2_1_processed.pdf'
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    groupby = "leiden_res_1_5"
)

In [ ]:
adata = adata[~adata.obs['leiden_res_1_5'].isin(['16'])]

In [ ]:
adata = adata[~adata.obs['timepoint'].isin(['2', '3', '4', '6', '7'])]

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'C1QA', color = 'leiden_res_1_5')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'MRC1', color = 'leiden_res_1_5')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'F13A1', color = 'leiden_res_1_5')

In [ ]:
sc.pl.scatter(adata, x = 'pct_counts_mt', y = 'LYVE1', color = 'leiden_res_1_5')

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    groupby = "leiden_res_2"
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_res_1", standard_scale="var", n_genes=30, dendrogram = None
    #, save = 'res1_top30genes.pdf'
)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_1", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_res_1_", standard_scale="var", n_genes=10, dendrogram = None#, save = 'res1_top30genes.pdf'
)

In [ ]:
sc.pl.dotplot(adata, ['C1QA','C1QC','CD68','PTPRC','S100A8','LYVE1', 'FOLR2', 'F13A1', 'SELENOP', 'BNC2', 'SAMD4A', 'SNTB2', 'PMP22','SLC18B1',
           'SLC2A13', 'PLPP3', 'ABCA1', 'STARD13', 'MYO1D', 'MRC1', 'DAAM2', 'SRGAP1',
           'FMN1', 'HRH1', 'TBC1D16'], 
              groupby = 'leiden_res_1_5', dendrogram = None#, save = 'lyve1keymarkers.pdf'
             )

In [ ]:
sc.pl.dotplot(adata, ['HLA-DRA', 'CD68', 'HLA-A', 'PSAP', 'AIF1', 'CLIC1', 'S100A9', 'LGALS1', 'PFN1', 'MYL6', 'VAMP8', 
                     'TSPO', 'BTF3', 'POLR2L', 'PSMA7', 'POMP', 'COX7C', 'NDUFA1', 'HCST', 'DBI', 'TMA7'], 
              groupby = 'leiden_res_1', dendrogram = None, save = 'HLAkeymarkers.pdf')

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"], groupby = 'leiden_res_1',
    jitter=0.4,
    multi_panel=True, save = 'qcdata.pdf'

)

In [ ]:
adata = adata[~adata.obs['timepoint'].isin(['day0'])]

In [ ]:
adata = adata[~adata.obs['leiden_res_1_5'].isin(['1', '16', '17', '18'])]

In [ ]:
adata

In [ ]:
adata.X = adata.layers['counts']

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 1,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_1'
)

In [ ]:
!pwd

In [ ]:
adata = sc.read_h5ad('/Users/yanjun/Desktop/scanpy/filtered.h5ad')

In [ ]:
# PAGA and diffusion maps
sc.tl.paga(adata, 'leiden_res_1_5')
scv.pl.paga(adata, basis='umap', threshold=0.4, max_edge_width=0.3, alpha=.4, title='',  
            figsize=(4.7,5), legend_loc='right margin')

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata.X = adata.layers['counts']

In [ ]:
sc.tl.diffmap(adata, n_comps = 30)
scv.pl.diffmap(adata, color='timepoint', legend_loc='right margin', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map')

In [ ]:
scv.pl.diffmap(adata, color='leiden_res_1_5', legend_loc='on data', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map')

In [ ]:
sc.pl.umap(adata1, color=["leiden_res_1", 'condition',"hash.ID"], save= 'filtered.pdf')

In [ ]:
sc.tl.diffmap(adata, n_comps = 30)
sc.pl.diffmap(adata, color = ['leiden_res_1']) #save = 'all_diffmap_scanpy_res2.5.pdf')

In [ ]:
scv.pl.diffmap(adata, color='leiden_res_1_5', legend_loc='right margin', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map')#, save = 'MDIX_diffmap_scv_res1.pdf')

In [ ]:
adata = sc.read_h5ad("/home/yc/Desktop/py/filtered.h5ad")
adata = adata[adata.obs['condition'].isin(['MDIX'])]

In [ ]:
import cellrank as cr

In [ ]:
sc.pl.umap(adata, color=["leiden_res_1", "hash.ID"], save = 'MDIXWITHMTHIGHCLUSTER.pdf')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_1", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden_res_1", standard_scale="var", n_genes=10, dendrogram = None, save = 'topgenes_mdixwithhimtclus.pdf'
)

In [ ]:
adata.layers["spliced"] = adata.X
adata.layers["unspliced"] = adata.X
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
sc.tl.diffmap(adata, n_comps = 30)
sc.pl.diffmap(adata)

In [ ]:
scv.pl.diffmap(adata, color='ct_pseudotime', legend_loc='right margin', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map'#, save = 'MDIXwo_diffmap_ct.pdf'
              )

In [ ]:
adata.layers["spliced"] = adata.X
adata.layers["unspliced"] = adata.X
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
from cellrank.kernels import CytoTRACEKernel
ctk = CytoTRACEKernel(adata).compute_cytotrace()

In [ ]:
adata

In [ ]:
sc.pl.embedding(
    adata,
    color=["ct_pseudotime", "leiden_res_1", "hash.ID"],
    basis="umap",
    color_map="gnuplot2"#, save = 'MDIXwo_ct.pdf'
)

In [ ]:
adata.write('mdixwo.h5ad')

In [ ]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="hash.ID", rotation=90,  save = 'vlnMDIXwo_ct.pdf')

In [ ]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="leiden_res_1", rotation=90, save = 'vlnMDIXwo_ct2.pdf')

In [ ]:
scv.pl.proportions(adata, groupby = 'leiden_res_1')

In [ ]:
adata.write('mdixwithhighmt.h5ad')

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
scv.pp.filter_and_normalize(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.tl.velocity(adata)

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
scv.pl.velocity_embedding(adata, basis='umap', frameon=False#, save='embedding.pdf'
                         )

In [ ]:
scv.pl.velocity_embedding(adata, basis='diffmap', frameon=False#, save='embedding.pdf'
                         )

In [ ]:
scv.pl.velocity(adata, ['C1QA','C1QC','CD68','PTPRC','S100A8','LYVE1', 'FOLR2', 'F13A1', 'SELENOP','BNC2', 'SAMD4A', 'SNTB2', 'PMP22','SLC18B1',
           'SLC2A13', 'PLPP3', 'ABCA1', 'STARD13', 'MYO1D', 'MRC1', 'DAAM2', 'SRGAP1',
           'FMN1', 'HRH1', 'TBC1D16'], ncols=2, basis = 'diffmap')

In [ ]:
adata.write['option3_1.h5ad']

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'leiden_res_1_5'#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']
scv.tl.paga(adata, groups='leiden_res_1_5')
df = scv.get_df(adata, 'paga/transitions_confidence', precision=2).T
df.style.background_gradient(cmap='Blues').format('{:.2g}')

In [ ]:
scv.pl.paga(adata, basis='diffmap', size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5)

In [ ]:
scv.pl.velocity_embedding_grid(adata, basis='umap', color='leiden_res_1', title='', scale=0.25)

In [ ]:
scv.pl.velocity(adata, ['LYVE1',  'C1QA', 'F13A1', 'APOE'], ncols=2)